# Vintage Analysis & Roll Rate Matrix — Bad Definition

This notebook determines the credit card **Bad** target variable through:
1. **Roll Rate Analysis** — month-to-month delinquency bucket migration probabilities
2. **Vintage Analysis** — cumulative bad-rate curves by approval cohort to identify the maturity point
3. **Bad Definition Comparison** — evaluates candidate definitions (Ever30DPD\_6M, Ever60DPD\_12M, Ever90DPD\_12M)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# All tuneable parameters live here so nothing is hard-coded deeper in the notebook.

# Date-range presets for data loading
DATE_RANGES = {
    # "full": {"start": "2022-06-30", "end": "2024-07-01", "mob_months": 25},
     "full": {"start": "2022-06-30", "end": "2025-07-01", "mob_months": 25},
    "recent": {"start": "2025-01-31", "end": "2026-07-01", "mob_months": 37},
}

# Bad-definition candidates (min_bucket follows the DELINQUENCYBUCKET encoding:
#   0=current, 1=1-29 DPD, 2=30-59 DPD, 3=60-89 DPD, 4=90+ DPD)
BAD_DEFINITIONS = {
    "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
    "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
    "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
}

# Target generation: 12-month performance window
TARGET_WINDOW_MONTHS = 12

# Minimum OPEN snapshots a card must have to be retained
MIN_OPEN_SNAPSHOTS = 3

# Columns needed downstream (avoids OOM when converting full 88-col frame to pandas)
ANALYSIS_COLS = [
    "SNAPSHOT_DATE", "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER", "FTS_Ref_No",
]

# Output directory for charts
OUTPUT_DIR = "/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/input/"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

## 1. Data Loading & Cleaning
Parameterised functions that load the credit card snapshot data, apply business
exclusion filters, generate the target variable, and merge AECB bureau features.

In [ ]:
def load_and_clean_credit_card_data(date_start, date_end, mob_months=37):
    """
    Load credit card history and apply business exclusion filters.

    Parameters
    ----------
    date_start : str   – APPROVEDATE lower bound (exclusive), e.g. '2022-06-30'
    date_end   : str   – APPROVEDATE upper bound (exclusive), e.g. '2025-07-01'
    mob_months : int   – months-on-book observation window (default 37)

    Returns
    -------
    pyspark.sql.DataFrame  – cleaned Spark DataFrame with exclusions applied.
    """
    # Step 1: Load raw data within the observation window
    df = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, {mob_months})
          AND APPROVEDATE > '{date_start}'
          AND APPROVEDATE < '{date_end}'
          AND UPPER(TRIM(CARD_BUSINESS_SEGMENT)) IN ('CARDS')
          AND UPPER(TRIM(CUSTOMER_SEGMENT)) = 'SALARIED'
    """)

    # Step 2: Count OPEN snapshots per card (needed for Exclusion 9)
    w = Window.partitionBy("CREDITCARDNO")
    df = df.withColumn(
        "open_count",
        F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w),
    )

    # Step 3: Exclusion conditions
    exclude_cond = (
        (F.col("VIP") == "1")                                                             # 3a: VIP customers
        | (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y")                             # 3b: Ruling family
        | (F.col("PRIMARYCARD") == 0)                                                      # 7: Supplementary cards
        | (F.col("open_count") < 3)                                                        # 9: Fewer than 3 OPEN
        | (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM")                        # 10a: Salaried-IM
        | (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
              "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))          # 10b: Unwanted segment policies
        |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('FRUD')))
        # |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('SKIP', 'SKP7', 'SKIP', 'SPSK', 'FRUD')))

    inclusive_cond = (
        (F.col("CLASSCODE").isNull()| ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))   # 1, 8: SME/STAFF class code
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))                  # 2: CARDS segment
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")                       # 6: Salaried
    )        

    df_clean = df.filter(~exclude_cond).filter(inclusive_cond)#.drop("open_count")

    # Step 4: Keep only "active" customers — at least 4 out of 12 snapshot-months
    #         with CUROUTBALTOTAL < 0 (i.e. carrying a balance, indicating usage).
    # w_card = Window.partitionBy("CREDITCARDNO")
    # df_clean = (
    #     df_clean
    #     .withColumn(
    #         "_active_months",
    #         F.sum(
    #             F.when(F.col("CUROUTBALTOTAL") < 0, 1).otherwise(0)
    #         ).over(w_card),
    #     )
    #     .filter(F.col("_active_months") >= 4)
    #     .drop("_active_months")
    # )

    return df_clean

In [ ]:
%sql
SELECT
  SUM(CASE WHEN CUROUTBALTOTAL > 0 THEN 1 ELSE 0 END)  AS cnt_positive,
  SUM(CASE WHEN CUROUTBALTOTAL < 0 THEN 1 ELSE 0 END)  AS cnt_negative,
  SUM(CASE WHEN CUROUTBALTOTAL = 0 THEN 1 ELSE 0 END)  AS cnt_zero,
  COUNT(*) AS total_rows
FROM mis_automation.silver_credit_card.vw_credit_card_history

In [ ]:
%sql
SELECT
  CURRENT_STATUS,
  COUNT(*) AS cnt
FROM mis_automation.silver_credit_card.vw_credit_card_history
GROUP BY CURRENT_STATUS
ORDER BY cnt DESC

In [ ]:
def filter_cards_with_sufficient_open_snapshots(df_pd, min_open=MIN_OPEN_SNAPSHOTS):
    """
    Retain only cards with >= `min_open` OPEN-status snapshots after all
    exclusion filters.  Addresses edge cases where employment-category
    changes (e.g. SALARIED -> SELF-EMPLOYED mid-observation) reduced the
    OPEN count below the threshold.

    Typically removes ~300 cards — negligible sample-size impact.
    """
    open_counts = (
        df_pd
        .groupby("CREDITCARDNO")["CURRENT_STATUS"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    qualifying_cards = open_counts.loc[
        open_counts["1.OPEN"] >= min_open, "CREDITCARDNO"
    ]
    df_out = df_pd.loc[df_pd["CREDITCARDNO"].isin(qualifying_cards.tolist())]

    print(f"Cards before filter : {df_pd['CREDITCARDNO'].nunique()}")
    print(f"Cards after filter  : {df_out['CREDITCARDNO'].nunique()}")
    print(f"Cards removed       : {df_pd['CREDITCARDNO'].nunique() - df_out['CREDITCARDNO'].nunique()}")
    print(f"Rows  before: {df_pd.shape[0]:,}  |  Rows after: {df_out.shape[0]:,}")
    return df_out

In [ ]:
def generate_target(df, window_months=TARGET_WINDOW_MONTHS):
    """
    Assign one target label per CREDITCARDNO based on the max
    DELINQUENCYBUCKET within the first `window_months` after approval.

    Target mapping
    --------------
    -2 : immature (fewer than `window_months` unique snapshot months)
     0 : good    (max bucket == 0)
    -1 : indeterminate (max bucket in 1-3)
     1 : bad     (max bucket >= 4, i.e. 90+ DPD)
    """
    df = df.copy()
    df["SNAPSHOT_DATE"] = pd.to_datetime(df["SNAPSHOT_DATE"])
    df["APPROVEDATE"]   = pd.to_datetime(df["APPROVEDATE"])

    mask = (
        (df["SNAPSHOT_DATE"] >= df["APPROVEDATE"])
        & (df["SNAPSHOT_DATE"] < df["APPROVEDATE"] + pd.DateOffset(months=window_months))
    )
    window_df = df.loc[mask].copy()
    window_df["_snap_month"] = window_df["SNAPSHOT_DATE"].dt.to_period("M")

    agg = (
        window_df
        .groupby("CREDITCARDNO")
        .agg(max_bucket=("DELINQUENCYBUCKET", "max"),
             unique_months=("_snap_month", "nunique"))
        .reset_index()
    )

    conditions = [
        agg["unique_months"] < window_months,  # -2: immature
        agg["max_bucket"] >= 4,                 #  1: bad 3 is used
        agg["max_bucket"] == 0,                 #  0: good
        agg["max_bucket"].between(1, 3),        # -1: indeterminate
    ]
    agg["target"] = np.select(conditions, [-2, 1, 0, -1], default=np.nan)

    if agg["target"].isna().any():
        n_drop = agg["target"].isna().sum()
        print(f"WARNING: {n_drop} rows matched no target condition — dropped.")
        agg = agg.dropna(subset=["target"])
    agg["target"] = agg["target"].astype(int)

    id_cols = ["CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE",
               "REFERENCE_NUMBER", "FTS_Ref_No"]
    id_df = df[id_cols].drop_duplicates(subset="CREDITCARDNO")
    result = id_df.merge(agg[["CREDITCARDNO", "target"]], on="CREDITCARDNO", how="inner")
    result["REFERENCE_NUMBER"] = (
        result["REFERENCE_NUMBER"].astype(str).str.replace(r'\.0$', '', regex=True)
    )
    return result

In [ ]:
def load_aecb_feature_store(df_target):
    """
    Load the AECB feature-store table, filter to reference numbers
    present in `df_target`, merge, and split into thin-file subset.

    Returns
    -------
    aecb_merged : pd.DataFrame – AECB features joined with target
    df_thin     : pd.DataFrame – subset where file_type == 'Thin File'
    """
    aecb = spark.sql("""
        SELECT * FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
    """)
    ref_numbers = [str(r) for r in df_target["REFERENCE_NUMBER"].unique().tolist()]
    aecb_filtered = (
        aecb.withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
            .filter(F.col("Reference_Number").isin(ref_numbers))
    )
    aecb_pd = aecb_filtered.toPandas()

    # Ensure consistent join-key dtypes
    aecb_pd["Reference_Number"] = aecb_pd["Reference_Number"].astype(str)
    df_target["REFERENCE_NUMBER"] = df_target["REFERENCE_NUMBER"].astype(str)

    aecb_merged = aecb_pd.merge(
        df_target[["REFERENCE_NUMBER", "target", "CREDITCARDNO", "CRNNO","CIF"]].drop_duplicates(),
        left_on="Reference_Number",
        right_on="REFERENCE_NUMBER",
        how="inner",
    )
    df_thin = aecb_merged.loc[aecb_merged["file_type_new"] == "Thin File"].reset_index(drop=True)#File_type_new vs file_type

    print(f"AECB shape   : {aecb_pd.shape}")
    print(f"Merged shape : {aecb_merged.shape}")
    print(f"Thin-file    : {df_thin.shape[0]:,} rows  |  {df_thin['CREDITCARDNO'].nunique():,} cards")
    print(f"\nTarget distribution:\n{aecb_merged['target'].value_counts(dropna=False).sort_index()}")
    return aecb_merged, df_thin

## 2. Analysis Functions
Reusable building blocks for roll-rate matrices, bad-definition flagging,
vintage curves, and the combined orchestrator.

In [ ]:
def prepare_data(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                 bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE"):
    """
    Clean dates and compute Months-on-Book (MOB).

    Adds columns: _snap_month, _approve_month, MOB.
    Drops rows where MOB < 0 (snapshot precedes approval).
    """
    df = df.copy()
    df[snap_col]    = pd.to_datetime(df[snap_col])
    df[approve_col] = pd.to_datetime(df[approve_col])

    df["_snap_month"]    = df[snap_col].values.astype("datetime64[M]")
    df["_approve_month"] = df[approve_col].values.astype("datetime64[M]")

    df["MOB"] = (
        (df["_snap_month"].dt.year  - df["_approve_month"].dt.year) * 12
        + (df["_snap_month"].dt.month - df["_approve_month"].dt.month)
    )

    before = len(df)
    df = df[df["MOB"] >= 0].copy()
    dropped = before - len(df)
    if dropped:
        print(f"[prepare_data] Dropped {dropped} rows with MOB < 0")

    return df.sort_values([id_col, snap_col]).reset_index(drop=True)

In [ ]:
def build_roll_rate_matrix(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                           bucket_col="DELINQUENCYBUCKET", n_periods=1):
    """
    Build an n-period delinquency transition matrix.

    Returns (transition_rate_pct, transition_counts).
    Only considers pairs separated by exactly `n_periods` months.
    """
    df = df.sort_values([id_col, snap_col]).copy()
    df["_period"] = df[snap_col].dt.to_period("M")

    df["_next_bucket"] = df.groupby(id_col)[bucket_col].shift(-1)
    df["_next_period"] = df.groupby(id_col)["_period"].shift(-1)

    valid = df.dropna(subset=["_next_bucket", "_next_period"]).copy()
    gap = (valid["_next_period"] - valid["_period"]).apply(lambda x: x.n)
    valid = valid[gap == n_periods]

    counts = pd.crosstab(valid[bucket_col], valid["_next_bucket"])
    rates  = counts.div(counts.sum(axis=1), axis=0) * 100
    return rates.round(2), counts


def build_worst_bucket_transition_matrix(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                                         bucket_col="DELINQUENCYBUCKET", horizon_months=6):
    """
    Build a transition matrix where the destination state is the **worst**
    (maximum) delinquency bucket reached within the next `horizon_months`.

    This answers: "Given a customer is in bucket X today, what is the
    probability they reach bucket 0/1/2/3/4 (worst case) within the
    next N months?"

    Returns (transition_rate_pct, transition_counts).
    """
    df = df.sort_values([id_col, snap_col]).copy()
    df[snap_col] = pd.to_datetime(df[snap_col])

    records = []
    for card_id, grp in df.groupby(id_col):
        grp = grp.reset_index(drop=True)
        for idx, row in grp.iterrows():
            current_bucket = row[bucket_col]
            current_date   = row[snap_col]
            horizon_end    = current_date + pd.DateOffset(months=horizon_months)

            future = grp[
                (grp[snap_col] > current_date) & (grp[snap_col] <= horizon_end)
            ]
            if future.empty:
                continue  # skip if no future data within horizon

            worst_bucket = future[bucket_col].max()
            records.append({
                "current_bucket": current_bucket,
                "worst_bucket":   worst_bucket,
            })

    if not records:
        return pd.DataFrame(), pd.DataFrame()

    result_df = pd.DataFrame(records)
    counts = pd.crosstab(result_df["current_bucket"], result_df["worst_bucket"])
    counts.index.name   = bucket_col
    counts.columns.name = f"Worst Bucket (next {horizon_months}m)"
    rates = counts.div(counts.sum(axis=1), axis=0) * 100
    return rates.round(2), counts


def plot_worst_bucket_heatmap(transition_rate, horizon_months=6, save_path=None):
    """Annotated heatmap for the worst-bucket transition matrix."""
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100)

    ax.set_xticks(range(len(transition_rate.columns)))
    ax.set_yticks(range(len(transition_rate.index)))
    ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns],
                       fontsize=10, fontweight="medium", rotation=45, ha="right")
    ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index],
                       fontsize=10, fontweight="medium")
    ax.tick_params(axis="x", bottom=True, top=False, labelbottom=True, labeltop=False, pad=8)
    ax.tick_params(axis="y", pad=5)
    ax.set_xlabel(f"Worst Bucket Reached (next {horizon_months} months)")
    ax.set_ylabel("Current Bucket")
    ax.set_title(f"Worst-Case Roll Rate Matrix ({horizon_months}-Month Horizon, %)")

    for i in range(transition_rate.shape[0]):
        for j in range(transition_rate.shape[1]):
            ax.text(j, i, f"{transition_rate.values[i, j]:.1f}",
                    ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, label="Transition Probability (%)")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def plot_roll_rate_heatmap(transition_rate, n_periods=1, save_path=None):
    """Annotated heatmap of the transition matrix."""
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100)

    ax.set_xticks(range(len(transition_rate.columns)))
    ax.set_yticks(range(len(transition_rate.index)))
    ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns],
                       fontsize=10, fontweight="medium", rotation=45, ha="right")
    ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index],
                       fontsize=10, fontweight="medium")
    ax.tick_params(axis="x", bottom=True, top=False, labelbottom=True, labeltop=False, pad=8)
    ax.tick_params(axis="y", pad=5)
    ax.set_xlabel(f"Bucket after {n_periods} month(s)")
    ax.set_ylabel("Current Bucket")
    ax.set_title(f"Roll Rate Matrix ({n_periods}-Month Transition, %)")

    for i in range(transition_rate.shape[0]):
        for j in range(transition_rate.shape[1]):
            ax.text(j, i, f"{transition_rate.values[i, j]:.1f}",
                    ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, label="Transition Probability (%)")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig

In [ ]:
def eventual_roll_forward_rate(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                               bucket_col="DELINQUENCYBUCKET",
                               from_bucket=2, to_bucket_min=4, within_months=6):
    """
    Among all account-months at `from_bucket`, what fraction eventually
    reach >= `to_bucket_min` within the next `within_months` months?

    High rates (e.g. >80 %) indicate `from_bucket` is already a strong
    bad signal and the bad definition could be brought forward.
    """
    df = df.sort_values([id_col, snap_col]).copy()
    hits, total = 0, 0
    for _, g in df.groupby(id_col):
        g = g.reset_index(drop=True)
        at_from = g[g[bucket_col] == from_bucket]
        for _, row in at_from.iterrows():
            future = g[
                (g[snap_col] > row[snap_col])
                & (g[snap_col] <= row[snap_col] + pd.DateOffset(months=within_months))
            ]
            if len(future) == 0:
                continue
            total += 1
            if (future[bucket_col] >= to_bucket_min).any():
                hits += 1
    rate = (hits / total * 100) if total > 0 else np.nan
    return rate, total

In [ ]:
def define_bad_flags(df, id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET",
                     mob_col="MOB", bad_definitions=None):
    """
    For each candidate bad definition, label every account as
    mature/immature and bad/good.  Returns (account_summary, bad_definitions).
    """
    if bad_definitions is None:
        bad_definitions = BAD_DEFINITIONS

    account_summary = df.groupby(id_col).agg(
        max_mob_observed=("MOB", "max"),
        approve_month=("_approve_month", "first"),
    ).reset_index()

    for name, cfg in bad_definitions.items():
        window     = cfg["window_months"]
        min_bucket = cfg["min_bucket"]

        account_summary[f"{name}_mature"] = account_summary["max_mob_observed"] >= window

        window_df = df[df[mob_col] <= window]
        bad_flags = (
            window_df.groupby(id_col)[bucket_col]
            .apply(lambda s: bool((s >= min_bucket).any()))
            .rename(f"{name}_bad")
        )
        account_summary = account_summary.merge(bad_flags, on=id_col, how="left")
        account_summary[f"{name}_bad"] = account_summary[f"{name}_bad"].fillna(False)

        account_summary[f"{name}_target"] = np.where(
            ~account_summary[f"{name}_mature"], -1,
            np.where(account_summary[f"{name}_bad"], 1, 0),
        )
    return account_summary, bad_definitions


def compare_bad_definitions(account_summary, bad_definitions):
    """
    Compare candidate bad definitions by mature-sample bad rate,
    pairwise agreement (%), and correlation.
    """
    names = list(bad_definitions.keys())

    # Per-definition summary
    rows = []
    for name in names:
        mature = account_summary[f"{name}_mature"]
        pop = account_summary[mature]
        bad_rate = pop[f"{name}_bad"].mean() * 100 if len(pop) else np.nan
        rows.append({"definition": name,
                     "mature_sample_size": int(mature.sum()),
                     "bad_rate_%": round(bad_rate, 2)})
    summary_df = pd.DataFrame(rows)

    # Pairwise overlap
    overlap_rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            n1, n2 = names[i], names[j]
            both = account_summary[f"{n1}_mature"] & account_summary[f"{n2}_mature"]
            sub  = account_summary[both]
            if len(sub) == 0:
                continue
            agree = (sub[f"{n1}_bad"] == sub[f"{n2}_bad"]).mean() * 100
            corr  = sub[f"{n1}_bad"].astype(int).corr(sub[f"{n2}_bad"].astype(int))
            overlap_rows.append({"pair": f"{n1} vs {n2}",
                                 "sample_size": len(sub),
                                 "agreement_%": round(agree, 2),
                                 "correlation": round(corr, 3)})
    overlap_df = pd.DataFrame(overlap_rows)
    return summary_df, overlap_df

In [ ]:
def vintage_cumulative_bad_rate(df, account_summary, id_col="CREDITCARDNO",
                                bucket_col="DELINQUENCYBUCKET", min_bucket=4,
                                mob_col="MOB", max_mob=36):
    """
    Pivot table: rows = vintage (approval month), cols = MOB,
    values = cumulative bad rate (%).
    """
    merged = df.merge(
        account_summary[[id_col, "approve_month"]], on=id_col, how="left"
    )
    merged["vintage"] = merged["approve_month"].dt.to_period("M").astype(str)
    merged = merged.sort_values([id_col, mob_col])
    merged["_is_bad"]  = merged[bucket_col] >= min_bucket
    merged["_cum_bad"] = merged.groupby(id_col)["_is_bad"].cummax()

    pivot = merged.pivot_table(
        index="vintage", columns=mob_col, values="_cum_bad", aggfunc="mean"
    ) * 100
    return pivot.loc[:, pivot.columns <= max_mob].round(2)


def plot_vintage_curves(pivot, title="Vintage Cumulative Bad Rate", save_path=None):
    """Line chart with one curve per vintage cohort."""
    fig, ax = plt.subplots(figsize=(10, 6))
    for vintage in pivot.index:
        s = pivot.loc[vintage].dropna()
        ax.plot(s.index, s.values, marker="o", markersize=3, label=vintage)
    ax.set_xlabel("Months on Book (MOB)")
    ax.set_ylabel("Cumulative Bad Rate (%)")
    ax.set_title(title)
    ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def find_maturity_point(pivot, marginal_threshold=0.3):
    """
    Detect the MOB where the average cumulative bad-rate curve
    flattens (marginal increase < threshold pp).  Returns
    (mature_mob, avg_curve, marginal_series).
    """
    avg = pivot.mean(axis=0, skipna=True).sort_index()
    marginal = avg.diff()
    candidates = marginal[(marginal.index > 1) & (marginal.abs() < marginal_threshold)]
    mature_mob = candidates.index.min() if len(candidates) else np.nan
    return mature_mob, avg, marginal

In [ ]:
def run_full_analysis(df, bad_definitions=None, output_dir=".",
                      id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                      bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE"):
    """
    End-to-end analysis pipeline:
      1. Roll-rate matrix (1-month) + heatmap
      2. Eventual roll-forward rate (30-59 DPD -> 90+ DPD within 6 months)
      3. Bad-definition labelling & comparison
      4. Vintage cumulative bad-rate curves & maturity point
    """
    df = prepare_data(df, id_col, snap_col, bucket_col, approve_col)

    # ---- 1. Roll Rate --------------------------------------------------------
    print("\n" + "=" * 60)
    print("1. ROLL RATE ANALYSIS")
    print("=" * 60)
    roll_1m, _ = build_roll_rate_matrix(df, id_col, snap_col, bucket_col, n_periods=1)
    print("1-month transition matrix (%):\n", roll_1m)
    plot_roll_rate_heatmap(roll_1m, n_periods=1,
                           save_path=f"{output_dir}/roll_rate_1m.png")

    rate, n_obs = eventual_roll_forward_rate(
        df, id_col, snap_col, bucket_col,
        from_bucket=2, to_bucket_min=4, within_months=6,
    )
    print(f"\n30-59 DPD -> 90+ DPD within 6 months: {rate:.1f}%  (n={n_obs:,})")

    # ---- 2. Bad Definition Comparison ----------------------------------------
    print("\n" + "=" * 60)
    print("2. BAD DEFINITION COMPARISON")
    print("=" * 60)
    account_summary, bad_defs = define_bad_flags(
        df, id_col, bucket_col, "MOB", bad_definitions,
    )
    summary_df, overlap_df = compare_bad_definitions(account_summary, bad_defs)
    print("\nBad-rate by definition (mature samples):")
    print(summary_df.to_string(index=False))
    print("\nPairwise agreement & correlation:")
    print(overlap_df.to_string(index=False))

    # ---- 3. Vintage Curves ---------------------------------------------------
    print("\n" + "=" * 60)
    print("3. VINTAGE ANALYSIS")
    print("=" * 60)
    pivot = vintage_cumulative_bad_rate(
        df, account_summary, id_col, bucket_col, min_bucket=4, max_mob=24,
    )
    plot_vintage_curves(
        pivot, title="Vintage Cumulative Bad Rate (90+ DPD, bucket>=4)",
        save_path=f"{output_dir}/vintage_curves.png",
    )
    mature_mob, avg_curve, _ = find_maturity_point(pivot)
    print(f"\nEstimated maturity MOB: {mature_mob}")
    print("\nAverage cumulative bad-rate by MOB:")
    print(avg_curve.round(2).to_string())

    return {
        "df_prepared": df,
        "roll_rate_1m": roll_1m,
        "account_summary": account_summary,
        "bad_definition_summary": summary_df,
        "bad_definition_overlap": overlap_df,
        "vintage_pivot": pivot,
        "maturity_mob": mature_mob,
    }

## 3. Execute — Full Date Range (Jul 2022 – Jun 2025)
Load, clean, generate targets, merge AECB features, then run the full
vintage & roll-rate analysis on **thin-file salaried cards**.

In [ ]:
cfg = DATE_RANGES["full"]
df_clean = load_and_clean_credit_card_data(cfg["start"], cfg["end"], cfg["mob_months"])
df_pd = df_clean.select(ANALYSIS_COLS).toPandas()
print(f"Columns: {ANALYSIS_COLS}")
print(f"Rows   : {df_pd.shape[0]:,}")

In [ ]:
df_clean_pd = filter_cards_with_sufficient_open_snapshots(df_pd)
df_clean_pd.shape

In [ ]:
df_target = generate_target(df_clean_pd)
print(f"Shape: {df_target.shape}")
print(f"\nTarget distribution:\n{df_target['target'].value_counts().sort_index()}")
df_target.head()

In [ ]:
df_thin['CRNNO'].nunique()#10076#22761#22761

In [ ]:
df_thin['target'].value_counts()
#  0    13845
# -2     3305
# -1     2165
#  1      746

In [ ]:
aecb_merged, df_thin = load_aecb_feature_store(df_target)

In [ ]:
pd.Series(df_thin["CIF"].unique(),name="CIF").to_csv("/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/cc_thin_file_aecb_cifs_1.csv")

In [ ]:
df_thin["CIF"].nunique()#10068
df_thin.shape#10611

In [ ]:
# --- Roll-Rate Analysis: First 6 Months vs Next 6 Months (Thin-File Customers) ---
# For each card, bucket the worst delinquency in MOB 1-6 ("from") and MOB 7-12 ("to") using inner join,
# then build a transition matrix to identify the right bad-rate definition.

_thin_cards_pd = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep = prepare_data(_thin_cards_pd)

# Worst delinquency bucket in first 6 months (MOB 1-6)
first_half = _thin_prep[_thin_prep["MOB"].between(1, 6)]
worst_first_6m = first_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_first_6m")

# Worst delinquency bucket in next 6 months (MOB 7-12)
second_half = _thin_prep[_thin_prep["MOB"].between(7, 12)]
worst_next_6m = second_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_next_6m")

# Inner join — keep only cards that appear in BOTH windows
transition_df = (
    worst_first_6m.to_frame()
    .join(worst_next_6m, how="inner")
)
transition_df["bucket_next_6m"] = transition_df["bucket_next_6m"].astype(int)

print(f"Cards with data in both 6-month windows (inner join): {len(transition_df):,}")

# Transition counts & rates
counts_6m = pd.crosstab(transition_df["bucket_first_6m"], transition_df["bucket_next_6m"],
                        rownames=["Worst Bucket (MOB 1-6)"], colnames=["Worst Bucket (MOB 7-12)"])
rates_6m = counts_6m.div(counts_6m.sum(axis=1), axis=0) * 100
rates_6m = rates_6m.round(2)

print("\n=== 6-Month Window Transition Matrix (%) ===")
print("From: Worst bucket in first 6 months  →  To: Worst bucket in next 6 months\n")
display(rates_6m)
print("\nTransition Counts:")
display(counts_6m)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rates_6m.values, cmap="Reds", vmin=0, vmax=100)

ax.set_xticks(range(len(rates_6m.columns)))
ax.set_yticks(range(len(rates_6m.index)))
ax.set_xticklabels([f"Bucket {c}" for c in rates_6m.columns], fontsize=10, fontweight="medium", rotation=45, ha="right")
ax.set_yticklabels([f"Bucket {r}" for r in rates_6m.index], fontsize=10, fontweight="medium")
ax.set_xlabel("Worst Bucket in Next 6 Months (MOB 7-12)")
ax.set_ylabel("Worst Bucket in First 6 Months (MOB 1-6)")
ax.set_title("Roll-Rate Matrix: First 6M vs Next 6M — Thin-File Cards (%)")

for i in range(rates_6m.shape[0]):
    for j in range(rates_6m.shape[1]):
        ax.text(j, i, f"{rates_6m.values[i, j]:.1f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Transition Probability (%)")
plt.tight_layout()
plt.show()

# --- Summary insight for bad-rate definition ---
print("\n=== Insight: Cure / No Change / Worsen Rates from Each Bucket ===")
print(f"{'Bucket':<10} {'Cure %':>10} {'No Change %':>14} {'Worsen %':>12}")
print("-" * 48)
for bucket in sorted(rates_6m.index):
    row = rates_6m.loc[bucket]
    cure      = row[row.index < bucket].sum()   # moved to a better (lower) bucket
    no_change = row[row.index == bucket].sum()             # stayed in the same bucket
    worse     = row[row.index > bucket].sum()    # moved to a worse (higher) bucket
    print(f"  {bucket:<8} {cure:>10.1f} {no_change:>14.1f} {worse:>12.1f}")

print("\n  Cure     = migrated to a lower (better) bucket in MOB 7-12")
print("  No Change = remained in the same bucket")
print("  Worsen  = migrated to a higher (worse) bucket in MOB 7-12")

In [ ]:
# --- Roll-Rate Analysis: First 6 Months vs Next 6 Months (Thin-File Customers) ---
# For each card, bucket the worst delinquency in MOB 1-6 ("from") and MOB 7-12 ("to"), fillna in MOB 7-12 with the last DELINQUENCYBUCKET in MOB 1-6,
# then build a transition matrix to identify the right bad-rate definition.

_thin_cards_pd = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep = prepare_data(_thin_cards_pd)

# Worst delinquency bucket in first 6 months (MOB 1-6)
first_half = _thin_prep[_thin_prep["MOB"].between(1, 6)]
worst_first_6m = first_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_first_6m")

# Worst delinquency bucket in next 6 months (MOB 7-12)
second_half = _thin_prep[_thin_prep["MOB"].between(7, 12)]
worst_next_6m = second_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_next_6m")

# For cards missing from MOB 7-12, use their last observed DELINQUENCYBUCKET
# in the first 6 months as a carry-forward proxy for the "to" bucket.
last_bucket_first_6m = (
    first_half.sort_values("MOB")
    .groupby("CREDITCARDNO")["DELINQUENCYBUCKET"]
    .last()
    .rename("last_bucket_first_6m")
)

# Left join — keep ALL cards from the first window
transition_df = (
    worst_first_6m.to_frame()
    .join(worst_next_6m)
    .join(last_bucket_first_6m)
)

# Fill missing MOB 7-12 bucket with last observed bucket from MOB 1-6
missing_mask = transition_df["bucket_next_6m"].isna()
transition_df["bucket_next_6m"] = transition_df["bucket_next_6m"].fillna(
    transition_df["last_bucket_first_6m"]
).astype(int)

print(f"Cards from first 6-month window: {len(transition_df):,}")
print(f"  ├─ With actual MOB 7-12 data : {(~missing_mask).sum():,}")
print(f"  └─ Carry-forward (last MOB 1-6): {missing_mask.sum():,}")

# Transition counts & rates
counts_6m = pd.crosstab(transition_df["bucket_first_6m"], transition_df["bucket_next_6m"],
                        rownames=["Worst Bucket (MOB 1-6)"], colnames=["Worst Bucket (MOB 7-12)"])
rates_6m = counts_6m.div(counts_6m.sum(axis=1), axis=0) * 100
rates_6m = rates_6m.round(2)

print("\n=== 6-Month Window Transition Matrix (%) ===")
print("From: Worst bucket in first 6 months  →  To: Worst bucket in next 6 months\n")
display(rates_6m)
print("\nTransition Counts:")
display(counts_6m)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rates_6m.values, cmap="Reds", vmin=0, vmax=100)

ax.set_xticks(range(len(rates_6m.columns)))
ax.set_yticks(range(len(rates_6m.index)))
ax.set_xticklabels([f"Bucket {c}" for c in rates_6m.columns], fontsize=10, fontweight="medium", rotation=45, ha="right")
ax.set_yticklabels([f"Bucket {r}" for r in rates_6m.index], fontsize=10, fontweight="medium")
ax.set_xlabel("Worst Bucket in Next 6 Months (MOB 7-12)")
ax.set_ylabel("Worst Bucket in First 6 Months (MOB 1-6)")
ax.set_title("Roll-Rate Matrix: First 6M vs Next 6M — Thin-File Cards (%)")

for i in range(rates_6m.shape[0]):
    for j in range(rates_6m.shape[1]):
        ax.text(j, i, f"{rates_6m.values[i, j]:.1f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Transition Probability (%)")
plt.tight_layout()
plt.show()

# --- Summary insight for bad-rate definition ---
print("\n=== Insight: Cure / No Change / Worsen Rates from Each Bucket ===")
print(f"{'Bucket':<10} {'Cure %':>10} {'No Change %':>14} {'Worsen %':>12}")
print("-" * 48)
for bucket in sorted(rates_6m.index):
    row = rates_6m.loc[bucket]
    cure      = row[row.index < bucket].sum()   # moved to a better (lower) bucket
    no_change = row[row.index == bucket].sum()             # stayed in the same bucket
    worse     = row[row.index > bucket].sum()    # moved to a worse (higher) bucket
    print(f"  {bucket:<8} {cure:>10.1f} {no_change:>14.1f} {worse:>12.1f}")

print("\n  Cure     = migrated to a lower (better) bucket in MOB 7-12")
print("  No Change = remained in the same bucket")
print("  Worsen  = migrated to a higher (worse) bucket in MOB 7-12")

In [ ]:
# --- Roll-Rate Analysis: First 6 Months vs Next 6 Months (Thin-File Customers) ---
# For each card, bucket the worst delinquency in MOB 1-6 ("from") and MOB 7-12 ("to"), fillna in MOB 7-12 with the last DELINQUENCYBUCKET in MOB 1-6,
# then build a transition matrix to identify the right bad-rate definition.

_thin_cards_pd = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep = prepare_data(_thin_cards_pd)

# Worst delinquency bucket in first 6 months (MOB 1-6)
first_half = _thin_prep[_thin_prep["MOB"].between(1, 6)]
worst_first_6m = first_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_first_6m")

# Worst delinquency bucket in next 6 months (MOB 7-12)
second_half = _thin_prep[_thin_prep["MOB"].between(7, 12)]
worst_next_6m = second_half.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_next_6m")

# For cards missing from MOB 7-12, use their last observed DELINQUENCYBUCKET
# in the first 6 months as a carry-forward proxy for the "to" bucket.
last_bucket_first_6m = (
    first_half.sort_values("MOB")
    .groupby("CREDITCARDNO")["DELINQUENCYBUCKET"]
    .last()
    .rename("last_bucket_first_6m")
)

# Left join — keep ALL cards from the first window
transition_df = (
    worst_first_6m.to_frame()
    .join(worst_next_6m)
    .join(last_bucket_first_6m)
)

# Fill missing MOB 7-12 bucket with last observed bucket from MOB 1-6
missing_mask = transition_df["bucket_next_6m"].isna()
transition_df["bucket_next_6m"] = transition_df["bucket_next_6m"].fillna(
    transition_df["last_bucket_first_6m"]
).astype(int)

print(f"Cards from first 6-month window: {len(transition_df):,}")
print(f"  ├─ With actual MOB 7-12 data : {(~missing_mask).sum():,}")
print(f"  └─ Carry-forward (last MOB 1-6): {missing_mask.sum():,}")

# Transition counts & rates
counts_6m = pd.crosstab(transition_df["bucket_first_6m"], transition_df["bucket_next_6m"],
                        rownames=["Worst Bucket (MOB 1-6)"], colnames=["Worst Bucket (MOB 7-12)"])
rates_6m = counts_6m.div(counts_6m.sum(axis=1), axis=0) * 100
rates_6m = rates_6m.round(2)

print("\n=== 6-Month Window Transition Matrix (%) ===")
print("From: Worst bucket in first 6 months  →  To: Worst bucket in next 6 months\n")
display(rates_6m)
print("\nTransition Counts:")
display(counts_6m)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rates_6m.values, cmap="Reds", vmin=0, vmax=100)

ax.set_xticks(range(len(rates_6m.columns)))
ax.set_yticks(range(len(rates_6m.index)))
ax.set_xticklabels([f"Bucket {c}" for c in rates_6m.columns], fontsize=10, fontweight="medium", rotation=45, ha="right")
ax.set_yticklabels([f"Bucket {r}" for r in rates_6m.index], fontsize=10, fontweight="medium")
ax.set_xlabel("Worst Bucket in Next 6 Months (MOB 7-12)")
ax.set_ylabel("Worst Bucket in First 6 Months (MOB 1-6)")
ax.set_title("Roll-Rate Matrix: First 6M vs Next 6M — Thin-File Cards (%)")

for i in range(rates_6m.shape[0]):
    for j in range(rates_6m.shape[1]):
        ax.text(j, i, f"{rates_6m.values[i, j]:.1f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Transition Probability (%)")
plt.tight_layout()
plt.show()

# --- Summary insight for bad-rate definition ---
print("\n=== Insight: Cure / No Change / Worsen Rates from Each Bucket ===")
print(f"{'Bucket':<10} {'Cure %':>10} {'No Change %':>14} {'Worsen %':>12}")
print("-" * 48)
for bucket in sorted(rates_6m.index):
    row = rates_6m.loc[bucket]
    cure      = row[row.index < bucket].sum()   # moved to a better (lower) bucket
    no_change = row[row.index == bucket].sum()             # stayed in the same bucket
    worse     = row[row.index > bucket].sum()    # moved to a worse (higher) bucket
    print(f"  {bucket:<8} {cure:>10.1f} {no_change:>14.1f} {worse:>12.1f}")

print("\n  Cure     = migrated to a lower (better) bucket in MOB 7-12")
print("  No Change = remained in the same bucket")
print("  Worsen  = migrated to a higher (worse) bucket in MOB 7-12")

In [ ]:
row = rates_6m.loc[3]
row
row[row.index == 3].sum() 

In [ ]:
# --- Roll-Rate Analysis: First 12 Months vs Next 12 Months (Thin-File Customers) ---
# For each card, bucket the worst delinquency in MOB 1-12 ("from") and MOB 13-24 ("to") using inner join,
# then build a transition matrix to identify the right bad-rate definition.

_thin_cards_pd_12 = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep_12 = prepare_data(_thin_cards_pd_12)

# Worst delinquency bucket in first 12 months (MOB 1-12)
first_half_12 = _thin_prep_12[_thin_prep_12["MOB"].between(1, 12)]
worst_first_12m = first_half_12.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_first_12m")

# Worst delinquency bucket in next 12 months (MOB 13-24)
second_half_12 = _thin_prep_12[_thin_prep_12["MOB"].between(13, 24)]
worst_next_12m = second_half_12.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_next_12m")

# Merge — keep only cards that have observations in BOTH windows
transition_df_12 = pd.merge(worst_first_12m, worst_next_12m, left_index=True, right_index=True)
print(f"Cards with data in both 12-month windows: {len(transition_df_12):,}")

# Transition counts & rates
counts_12m = pd.crosstab(transition_df_12["bucket_first_12m"], transition_df_12["bucket_next_12m"],
                         rownames=["Worst Bucket (MOB 1-12)"], colnames=["Worst Bucket (MOB 13-24)"])
rates_12m = counts_12m.div(counts_12m.sum(axis=1), axis=0) * 100
rates_12m = rates_12m.round(2)

print("\n=== 12-Month Window Transition Matrix (%) ===")
print("From: Worst bucket in first 12 months  →  To: Worst bucket in next 12 months\n")
display(rates_12m)
print("\nTransition Counts:")
display(counts_12m)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rates_12m.values, cmap="Reds", vmin=0, vmax=100)

ax.set_xticks(range(len(rates_12m.columns)))
ax.set_yticks(range(len(rates_12m.index)))
ax.set_xticklabels([f"Bucket {c}" for c in rates_12m.columns], fontsize=10, fontweight="medium", rotation=45, ha="right")
ax.set_yticklabels([f"Bucket {r}" for r in rates_12m.index], fontsize=10, fontweight="medium")
ax.set_xlabel("Worst Bucket in Next 12 Months (MOB 13-24)")
ax.set_ylabel("Worst Bucket in First 12 Months (MOB 1-12)")
ax.set_title("Roll-Rate Matrix: First 12M vs Next 12M — Thin-File Cards (%)")

for i in range(rates_12m.shape[0]):
    for j in range(rates_12m.shape[1]):
        ax.text(j, i, f"{rates_12m.values[i, j]:.1f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Transition Probability (%)")
plt.tight_layout()
plt.show()

# --- Summary insight for bad-rate definition ---
print("\n=== Insight: Cure / No Change / Worsen Rates from Each Bucket ===")
print(f"{'Bucket':<10} {'Cure %':>10} {'No Change %':>14} {'Worsen %':>12}")
print("-" * 48)
for bucket in sorted(rates_12m.index):
    row = rates_12m.loc[bucket]
    cure      = row[row.index < bucket].sum()   # moved to a better (lower) bucket
    no_change = row.get(bucket, 0.0)             # stayed in the same bucket
    worse     = row[row.index > bucket].sum()    # moved to a worse (higher) bucket
    print(f"  {bucket:<8} {cure:>10.1f} {no_change:>14.1f} {worse:>12.1f}")

print("\n  Cure     = migrated to a lower (better) bucket in MOB 13-24")
print("  No Change = remained in the same bucket")
print("  Worsen  = migrated to a higher (worse) bucket in MOB 13-24")

In [ ]:
# --- Roll-Rate Analysis: First 12 Months vs Next 12 Months (Thin-File Customers) ---
# For each card, bucket the worst delinquency in MOB 1-12 ("from") and MOB 13-24 ("to"),
# then build a transition matrix to identify the right bad-rate definition.

_thin_cards_pd_12 = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep_12 = prepare_data(_thin_cards_pd_12)

# Worst delinquency bucket in first 12 months (MOB 1-12)
first_half_12 = _thin_prep_12[_thin_prep_12["MOB"].between(1, 12)]
worst_first_12m = first_half_12.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_first_12m")

# Worst delinquency bucket in next 12 months (MOB 13-24)
second_half_12 = _thin_prep_12[_thin_prep_12["MOB"].between(13, 24)]
worst_next_12m = second_half_12.groupby("CREDITCARDNO")["DELINQUENCYBUCKET"].max().rename("bucket_next_12m")

# For cards missing from MOB 13-24, use their last observed DELINQUENCYBUCKET
# in the first 12 months as a carry-forward proxy for the "to" bucket.
last_bucket_first_12m = (
    first_half_12.sort_values("MOB")
    .groupby("CREDITCARDNO")["DELINQUENCYBUCKET"]
    .last()
    .rename("last_bucket_first_12m")
)

# Left join — keep ALL cards from the first window
transition_df_12 = (
    worst_first_12m.to_frame()
    .join(worst_next_12m)
    .join(last_bucket_first_12m)
)

# Fill missing MOB 13-24 bucket with last observed bucket from MOB 1-12
missing_mask_12 = transition_df_12["bucket_next_12m"].isna()
transition_df_12["bucket_next_12m"] = transition_df_12["bucket_next_12m"].fillna(
    transition_df_12["last_bucket_first_12m"]
).astype(int)

print(f"Cards from first 12-month window: {len(transition_df_12):,}")
print(f"  ├─ With actual MOB 13-24 data : {(~missing_mask_12).sum():,}")
print(f"  └─ Carry-forward (last MOB 1-12): {missing_mask_12.sum():,}")

# Transition counts & rates
counts_12m = pd.crosstab(transition_df_12["bucket_first_12m"], transition_df_12["bucket_next_12m"],
                         rownames=["Worst Bucket (MOB 1-12)"], colnames=["Worst Bucket (MOB 13-24)"])
rates_12m = counts_12m.div(counts_12m.sum(axis=1), axis=0) * 100
rates_12m = rates_12m.round(2)

print("\n=== 12-Month Window Transition Matrix (%) ===")
print("From: Worst bucket in first 12 months  →  To: Worst bucket in next 12 months\n")
display(rates_12m)
print("\nTransition Counts:")
display(counts_12m)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rates_12m.values, cmap="Reds", vmin=0, vmax=100)

ax.set_xticks(range(len(rates_12m.columns)))
ax.set_yticks(range(len(rates_12m.index)))
ax.set_xticklabels([f"Bucket {c}" for c in rates_12m.columns], fontsize=10, fontweight="medium", rotation=45, ha="right")
ax.set_yticklabels([f"Bucket {r}" for r in rates_12m.index], fontsize=10, fontweight="medium")
ax.set_xlabel("Worst Bucket in Next 12 Months (MOB 13-24)")
ax.set_ylabel("Worst Bucket in First 12 Months (MOB 1-12)")
ax.set_title("Roll-Rate Matrix: First 12M vs Next 12M — Thin-File Cards (%)")

for i in range(rates_12m.shape[0]):
    for j in range(rates_12m.shape[1]):
        ax.text(j, i, f"{rates_12m.values[i, j]:.1f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Transition Probability (%)")
plt.tight_layout()
plt.show()

# --- Summary insight for bad-rate definition ---
print("\n=== Insight: Cure / No Change / Worsen Rates from Each Bucket ===")
print(f"{'Bucket':<10} {'Cure %':>10} {'No Change %':>14} {'Worsen %':>12}")
print("-" * 48)
for bucket in sorted(rates_12m.index):
    row = rates_12m.loc[bucket]
    cure      = row[row.index < bucket].sum()   # moved to a better (lower) bucket
    no_change = row[row.index == bucket].sum()             # stayed in the same bucket
    worse     = row[row.index > bucket].sum()    # moved to a worse (higher) bucket
    print(f"  {bucket:<8} {cure:>10.1f} {no_change:>14.1f} {worse:>12.1f}")

print("\n  Cure     = migrated to a lower (better) bucket in MOB 13-24")
print("  No Change = remained in the same bucket")
print("  Worsen  = migrated to a higher (worse) bucket in MOB 13-24")

In [ ]:
# --- Thin-File: Total, Bad Count & Bad Rate by Approve Month ---

# Ensure APPROVE_MONTH exists
if "APPROVE_MONTH" not in df_thin.columns:
    df_thin["APPROVE_MONTH"] = pd.to_datetime(df_thin["APPROVEDATE"]).dt.to_period("M").astype(str)

# Aggregate per approve month
monthly = (
    df_thin.groupby("APPROVE_MONTH")
    .agg(total=("CREDITCARDNO", "count"), bad=("BAD_FLAG", "sum"))
    .reset_index()
)
monthly["bad_pct"] = (monthly["bad"] / monthly["total"] * 100).round(2)
monthly = monthly.sort_values("APPROVE_MONTH")

print(monthly.to_string(index=False))

# --- Plot ---
fig, ax1 = plt.subplots(figsize=(14, 6))

x = range(len(monthly))
width = 0.35

# Bar: Total & Bad counts
bars_total = ax1.bar([i - width/2 for i in x], monthly["total"], width, label="Total", color="steelblue", alpha=0.8)
bars_bad   = ax1.bar([i + width/2 for i in x], monthly["bad"],   width, label="Bad",   color="salmon",    alpha=0.8)

ax1.set_xlabel("Approve Month", fontsize=12)
ax1.set_ylabel("Count", fontsize=12)
ax1.set_xticks(list(x))
ax1.set_xticklabels(monthly["APPROVE_MONTH"], rotation=45, ha="right", fontsize=9)
ax1.legend(loc="upper left")

# Annotate bar values
for bar in bars_total:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=7)
for bar in bars_bad:
    if bar.get_height() > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=7)

# Line: Bad rate % on secondary axis
ax2 = ax1.twinx()
ax2.plot(list(x), monthly["bad_pct"].values, color="darkred", marker="o", linewidth=2, label="Bad Rate %")
ax2.set_ylabel("Bad Rate (%)", fontsize=12, color="darkred")
ax2.tick_params(axis="y", labelcolor="darkred")
ax2.legend(loc="upper right")

# Annotate bad rate values
for i, pct in enumerate(monthly["bad_pct"].values):
    ax2.annotate(f"{pct:.1f}%", (i, pct), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=8, color="darkred", fontweight="bold")

ax1.set_title("Thin-File Cards: Total vs Bad Count & Bad Rate by Approve Month", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# --- Sample cards present in MOB 1-6 but MISSING from MOB 7-12 ---
# Show their full snapshot-level records so we can understand why they disappear.

_thin_cards_pd = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prep = prepare_data(_thin_cards_pd)

# Cards that appear in MOB 1-6
cards_first_half = set(_thin_prep[_thin_prep["MOB"].between(1, 6)]["CREDITCARDNO"].unique())
# Cards that appear in MOB 7-12
cards_second_half = set(_thin_prep[_thin_prep["MOB"].between(7, 12)]["CREDITCARDNO"].unique())

# Cards in first half but NOT in second half
missing_cards = sorted(cards_first_half - cards_second_half)
print(f"Cards with MOB 1-6 data but NO MOB 7-12 data: {len(missing_cards):,}")
print(f"Cards with MOB 1-6 data: {len(cards_first_half):,}")
print(f"Cards with MOB 7-12 data: {len(cards_second_half):,}")

# Pick a few sample cards and show all their snapshot records
N_SAMPLES = 5
sample_cards = missing_cards[:N_SAMPLES]
print(f"\n=== Showing ALL snapshot records for {len(sample_cards)} sample cards ===")

for card in sample_cards:
    card_data = _thin_prep[_thin_prep["CREDITCARDNO"] == card].sort_values("MOB")
    print(f"\n{'='*80}")
    print(f"Card: {card}  |  Total snapshots: {len(card_data)}  |  MOB range: {card_data['MOB'].min()} – {card_data['MOB'].max()}")
    print(f"{'='*80}")
    display(card_data[["CREDITCARDNO", "MOB", "DELINQUENCYBUCKET", "_snap_month",
                       *[c for c in card_data.columns if c not in ["CREDITCARDNO", "MOB", "DELINQUENCYBUCKET", "_snap_month"]]]])


In [ ]:
df_clean.filter(F.col("CREDITCARDNO") == "5196306001991227").orderBy("SNAPSHOT_DATE").toPandas()

In [ ]:
df_thin.head()

In [ ]:
# Filter snapshot-level data to thin-file cards and prepare
_thin_cards_rr = df_thin.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prepared_rr = prepare_data(_thin_cards_rr)
print(f"Thin-file snapshot rows: {len(_thin_prepared_rr):,}  |  Cards: {_thin_prepared_rr['CREDITCARDNO'].nunique():,}")

# Build worst-bucket transition matrix (6-month horizon)
worst_rates_thin, worst_counts_thin = build_worst_bucket_transition_matrix(
    _thin_prepared_rr, horizon_months=6
)

print("\n=== Worst-Case Roll Rate Matrix (%) — Thin-File, 6-Month Horizon ===")
display(worst_rates_thin)
print("\nTransition Counts:")
display(worst_counts_thin)

# Plot heatmap
fig = plot_worst_bucket_heatmap(
    worst_rates_thin, horizon_months=6,
    save_path=f"{OUTPUT_DIR}worst_bucket_heatmap_thin.png"
)
plt.show()

## 4. Execute — Recent Date Range (Feb 2025 – Jun 2026)
Same pipeline with a narrower, more recent cohort for out-of-time comparison.

In [ ]:
cfg_r = DATE_RANGES["recent"]
df_clean_r = load_and_clean_credit_card_data(cfg_r["start"], cfg_r["end"], cfg_r["mob_months"])
df_pd_r = df_clean_r.select(ANALYSIS_COLS).toPandas()
print(f"Rows: {df_pd_r.shape[0]:,}")

In [ ]:
df_clean_pd_r = filter_cards_with_sufficient_open_snapshots(df_pd_r)
df_target_r   = generate_target(df_clean_pd_r)
aecb_merged_r, df_thin_r = load_aecb_feature_store(df_target_r)
print(f"\nTarget distribution (recent):\n{aecb_merged_r['target'].value_counts(dropna=False).sort_index()}")

## 5. Vintage Deep-Dive
Re-plot vintage curves with **snapshot date** on the x-axis (instead of MOB)
to reveal external / macro-economic shocks, and investigate outlier vintages.

In [ ]:
# Prepare thin-file cards from the full-range data
_thin_cards     = df_pd.loc[df_pd["CREDITCARDNO"].isin(df_thin["CREDITCARDNO"])].copy()
_thin_prepared  = prepare_data(_thin_cards)
_account_summary, _bad_defs = define_bad_flags(_thin_prepared)

# Cumulative bad-rate pivot (90+ DPD)
_vintage_pivot = vintage_cumulative_bad_rate(
    _thin_prepared, _account_summary,
    id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET",
    min_bucket=4, max_mob=36,
)

# Map (vintage, MOB) -> snapshot year-month for x-axis
_merged = _thin_prepared.merge(
    _account_summary[["CREDITCARDNO", "approve_month"]], on="CREDITCARDNO", how="left"
)
_merged["vintage"] = _merged["approve_month"].dt.to_period("M").astype(str)
_snap_lookup = (
    _merged.groupby(["vintage", "MOB"])["_snap_month"]
    .first().reset_index()
)
_snap_lookup["snap_ym"] = pd.to_datetime(_snap_lookup["_snap_month"]).dt.to_period("M").astype(str)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
for vintage in _vintage_pivot.index:
    series = _vintage_pivot.loc[vintage].dropna()
    mob_to_snap = _snap_lookup[_snap_lookup["vintage"] == vintage].set_index("MOB")["snap_ym"]
    x_labels = [mob_to_snap.get(m) for m in series.index]
    valid = [(lbl, val) for lbl, val in zip(x_labels, series.values) if lbl is not None]
    if not valid:
        continue
    labels, values = zip(*valid)
    ax.plot(labels, values, marker="o", markersize=3, label=vintage)

ax.set_xlabel("Snapshot Date (Year-Month)")
ax.set_ylabel("Cumulative Bad Rate (%)")
ax.set_title("Vintage Cumulative Bad Rate – Thin File Cards (90+ DPD)\nX-axis = Snapshot Month")
ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Identify which vintage has the highest cumulative bad rate
final_rates = _vintage_pivot.max(axis=1).sort_values(ascending=False)
print("=== Final Cumulative Bad Rate (%) per Vintage (descending) ===")
print(final_rates.to_string())

outlier_vintage = final_rates.index[0]
print(f"\n>>> Outlier vintage: {outlier_vintage}  |  Bad rate: {final_rates.iloc[0]:.2f}%")

# Card counts & bad rates per vintage
_merged_check = _thin_prepared.merge(
    _account_summary[["CREDITCARDNO", "approve_month"]], on="CREDITCARDNO", how="left"
)
_merged_check["vintage"] = _merged_check["approve_month"].dt.to_period("M").astype(str)

vintage_card_counts = (
    _merged_check.groupby("vintage")["CREDITCARDNO"].nunique()
    .reset_index().rename(columns={"CREDITCARDNO": "total_cards"})
)
bad_cards = (
    _merged_check[_merged_check["DELINQUENCYBUCKET"] >= 4]
    .groupby("vintage")["CREDITCARDNO"].nunique()
    .reset_index().rename(columns={"CREDITCARDNO": "bad_cards"})
)
vintage_summary = vintage_card_counts.merge(bad_cards, on="vintage", how="left").fillna(0)
vintage_summary["bad_cards"]  = vintage_summary["bad_cards"].astype(int)
vintage_summary["bad_rate_%"] = (vintage_summary["bad_cards"] / vintage_summary["total_cards"] * 100).round(2)
vintage_summary = vintage_summary.sort_values("bad_rate_%", ascending=False)
print("\n=== Vintage Summary: Card Counts & Bad Rates ===")
display(vintage_summary)

# Delinquency bucket distribution for the outlier
outlier_data = _merged_check[_merged_check["vintage"] == outlier_vintage]
print(f"\n=== Delinquency Bucket Distribution for Vintage {outlier_vintage} ===")
print(outlier_data["DELINQUENCYBUCKET"].value_counts().sort_index())

# MOB when cards first hit 90+ DPD for the outlier
outlier_mob_bad = (
    outlier_data[outlier_data["DELINQUENCYBUCKET"] >= 4]
    .groupby("MOB")["CREDITCARDNO"].nunique()
    .reset_index().rename(columns={"CREDITCARDNO": "cards_at_90plus"})
)
print(f"\n=== MOB when cards first hit 90+ DPD (Vintage {outlier_vintage}) ===")
print(outlier_mob_bad.to_string(index=False))